#### An AI agent to automate research : it will take a question, search the web, read the top results, and give the most relevant passages back along with a short summary

##### The core idea isn’t just to find pages with the right keywords, but to find passages with the right meaning, For his, use vector embeddings - a coordinate of meaning in a high-dumensional space. 

##### The sentence-transformers library provides models that are expert at tuning any piece of text into a list of numbers - a vector, that represents its location in that "meaningful space"

In [5]:
# All needed imports 

# Data Science and NLP Libraries
from bs4 import BeautifulSoup
from ddgs import DDGS  # DDGS library allows you to perform web searches directly from Python
from huggingface_hub import InferenceClient
import numpy as np
from sentence_transformers import SentenceTransformer

# Other Libraries
import re
import os
import time
import urllib.parse
import requests

In [3]:
# Testing the models, cause we need a free tier model
client = InferenceClient(token=os.getenv("HF_TOKEN"))

models_to_test = [
    "meta-llama/Meta-Llama-3-8B-Instruct",
    "meta-llama/Meta-Llama-3.1-8B-Instruct",
    "mistralai/Mistral-7B-Instruct-v0.3",
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen2.5-7B-Instruct",
    "microsoft/Phi-3-mini-4k-instruct",
    "google/gemma-2-9b-it",
    "HuggingFaceH4/zephyr-7b-beta",
]

for m in models_to_test:
    try:
        result = client.chat_completion(
            messages=[{"role": "user", "content": "Say hello"}],
            model=m,
            max_tokens=10
        )
        print(f"{m}: WORKS - {result.choices[0].message.content}")
    except Exception as e:
        print(f"{m}: FAILED - {str(e)[:80]}")

meta-llama/Meta-Llama-3-8B-Instruct: FAILED - (Request ID: Root=1-6a68c431-6a30c332503cee2d35da3e19;32451ffc-59d8-434c-81ae-52
meta-llama/Meta-Llama-3.1-8B-Instruct: FAILED - (Request ID: Root=1-6a68c431-6a3bad407155286756c12a27;9fd34f24-301b-43a5-b4cd-ac
mistralai/Mistral-7B-Instruct-v0.3: FAILED - (Request ID: Root=1-6a68c431-33150b7707b35d5255b81851;26faa54a-6c64-4bcd-a54d-57
mistralai/Mixtral-8x7B-Instruct-v0.1: FAILED - (Request ID: Root=1-6a68c432-7f4b05f033f3081b24736675;7620a0b8-867b-45e3-a57c-a3
Qwen/Qwen2.5-7B-Instruct: WORKS - Hello! How can I assist you today?
microsoft/Phi-3-mini-4k-instruct: FAILED - (Request ID: Root=1-6a68c432-337c4cbf52385a170cdff9e5;0a5b9174-737b-4575-a1eb-92
google/gemma-2-9b-it: FAILED - (Request ID: Root=1-6a68c432-3992d12511d990f464903aa4;15b52961-893a-43c3-8d41-f6
HuggingFaceH4/zephyr-7b-beta: FAILED - (Request ID: Root=1-6a68c432-25d687ec548876fa5058bcd6;e67b70ac-0550-4577-b164-d9


In [4]:
# Configuration and Settings
SEARCH_RESULTS = 6        # How many URLs to check
PASSAGES_PER_PAGE = 4     # How many passages to pull from each URL
TOP_PASSAGES = 5          # How many relevant passages to use for the summary
SUMMARY_SENTENCES = 3     # How many sentences in the final summary
TIMEOUT = 8               # How long to wait for a webpage to load
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2" # Fast, high-quality model

llm_client = InferenceClient(token=os.getenv("HF_TOKEN"))

In [6]:
# Web Search with DuckDuckGo
def unwrap_ddg(url):
    """Extracts real URL from DuckDuckGo redirect wrapper."""
    try:
        parsed = urllib.parse.urlparse(url)
        if "duckduckgo.com" in parsed.netloc:
            qs = urllib.parse.parse_qs(parsed.query)
            uddg = qs.get("uddg")
            if uddg:
                return urllib.parse.unquote(uddg[0])
    except Exception:
        pass
    return url

def search_web(query, max_results=SEARCH_RESULTS):
    """Searches the web and return a list of URLs."""
    urls = []
    with DDGS() as ddgs:
        for r in ddgs.text(query, max_results=max_results):
            url = r.get("href") or r.get("url")
            if not url:
                continue
            url = unwrap_ddg(url) # Clean up DDG redirect links
            urls.append(url)
    return urls

In [7]:
# Fetch and Clean the Web Pages with python libraries: requests and BeautifulSoup

def fetch_text(url, timeout=TIMEOUT):
    """Fetches and cleans text content from a URL."""
    headers = {"User-Agent": "Mozilla/5.0 (research-agent)"}
    try:
        r = requests.get(url, timeout=timeout, headers=headers, allow_redirects=True)
        if r.status_code != 200:
            return ""
        ct = r.headers.get("content-type", "")
        if "html" not in ct.lower(): # Skipping non-HTML content
            return ""
        
        soup = BeautifulSoup(r.text, "html.parser")
        
        # Removing all annoying tags
        for tag in soup(["script", "style", "noscript", "header", "footer", "svg", "iframe", "nav", "aside"]):
            tag.extract()
            
        # Get all paragraph text
        paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
        text = " ".join([p for p in paragraphs if p])
        
        if text.strip():
            # Cleaning up whitespace
            return re.sub(r"\s+", " ", text).strip()
            
        # --- Fallback logic if <p> tags fail ---
        meta = soup.find("meta", attrs={"name": "description"}) or soup.find("meta", attrs={"property": "og:description"})
        if meta and meta.get("content"):
            return meta["content"].strip()
        if soup.title and soup.title.string:
            return soup.title.string.strip()
            
    except Exception:
        return "" # Fails silently
    return ""

In [8]:
# Chunking, embedding, and ranking the text passages - breaking long articles into smaller passges and embedding them using SentenceTransformers, then ranking them based on their relevance to the query.
def chunk_passages(text, max_words=120):
    """Splits long text into smaller passages."""
    words = text.split()
    if not words:
        return []
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i : i + max_words]
        chunks.append(" ".join(chunk))
        i += max_words
    return chunks

def split_sentences(text):
    """Splits text into sentences."""
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]

def cosine(a, b):
    """Computes cosine similarity between two vectors."""
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)


In [9]:
# Research Agent
class ResearchAgent:
    def __init__(self, embed_model=EMBEDDING_MODEL):
        print(f"Loading embedder: {embed_model}...")
        self.embedder = SentenceTransformer(embed_model)

    def run(self, query, use_llm_summary=False):
        start = time.time()
        
        # Starting the search
        urls = search_web(query)
        print(f"Found {len(urls)} urls.")
        
        # Fetch & Chunk
        docs = []
        for u in urls:
            txt = fetch_text(u)
            if not txt:
                continue
            chunks = chunk_passages(txt, max_words=120)
            for c in chunks[:PASSAGES_PER_PAGE]:
                docs.append({"url": u, "passage": c})
        
        if not docs:
            print("No documents fetched.")
            return {"query": query, "passages": [], "summary": ""}
        
        # Embedding
        print(f"Embedding {len(docs)} passages...")
        texts = [d["passage"] for d in docs]
        emb_texts = self.embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
        q_emb = self.embedder.encode([query], convert_to_numpy=True)[0]

        # Ranking by similarity
        sims = [cosine(e, q_emb) for e in emb_texts]
        top_idx = np.argsort(sims)[::-1][:TOP_PASSAGES]
        top_passages = [{
            "url": docs[i]["url"],
            "passage": docs[i]["passage"],
            "score": float(sims[i])
        } for i in top_idx]

        # Summarization
        if use_llm_summary:
            summary = self._llm_summary(query, top_passages)
        else:
            summary = self._extractive_summary(query, q_emb, top_passages)

        elapsed = time.time() - start
        return {
            "query": query,
            "passages": top_passages,
            "summary": summary,
            "time": elapsed
        }
    
    def _extractive_summary(self, query, q_emb, top_passages):
        """generates summary using extractive method (no LLM needed)."""
        sentences = []
        for tp in top_passages:
            for s in split_sentences(tp["passage"]):
                sentences.append({"sent": s, "url": tp["url"]})

        if not sentences:
            return "No summary could be generated."

        sent_texts = [s["sent"] for s in sentences]
        sent_embs = self.embedder.encode(sent_texts, convert_to_numpy=True, show_progress_bar=False)
        sent_sims = [cosine(e, q_emb) for e in sent_embs]

        top_sent_idx = np.argsort(sent_sims)[::-1][:SUMMARY_SENTENCES]
        chosen = [sentences[idx] for idx in top_sent_idx]

        seen = set()
        lines = []
        for s in chosen:
            key = s["sent"].lower()[:80]
            if key in seen:
                continue
            seen.add(key)
            lines.append(f"{s['sent']} (Source: {s['url']})")
        return " ".join(lines)

    def _llm_summary(self, query, top_passages):
        """Generates summary using remote LLM."""
        context = "\n\n".join(
            f"[Source: {p['url']}]\n{p['passage']}" for p in top_passages
        )
        result = llm_client.chat_completion(
            messages=[{
                "role": "user",
                    "content": f"""Based on the research passages below, provide a clear, 
                    concise summary that answers the question. Cite the sources.

                    Question: {query}

                    Research Passages:
                    {context}

                    Summary:"""
                }],
                model=LLM_MODEL,
                max_tokens=300
            )
        return result.choices[0].message.content.strip()


In [10]:
agent = ResearchAgent()

q = "What causes the long heat waves in Europe and how they originate?"
print(f"\nResearching: {q}\n")
out = agent.run(q, use_llm_summary=True)

print("\nTop passages:")
for p in out["passages"]:
    print(f"  score {p['score']:.3f} | {p['url']}")
    print(f"  {p['passage'][:150]}...\n")

print("--- Summary ---")
print(out["summary"])
print("---------------")
print(f"\nDone in {out['time']:.1f}s")

Loading embedder: sentence-transformers/all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2326.68it/s]



Researching: What causes the long heat waves in Europe and how they originate?

Found 6 urls.
Embedding 20 passages...

Top passages:
  score 0.590 | https://www.france24.com/en/live-news/20260623-what-is-driving-europe-s-heatwave
  Paris (France) (AFP) – Europe is baking under a scorching heatwave, with health warnings in place across western and central parts of the continent as...

  score 0.585 | https://theconversation.com/europe-is-battling-a-record-breaking-heatwave-whats-making-it-so-severe-286019
  two main reasons. 1. Timing In Europe, the hottest time of year comes in mid- to late July , about a month after the summer solstice. However, recent ...

  score 0.576 | https://www.dw.com/en/why-europe-is-getting-so-hot/a-77320871
  Much of Western Europe is suffering through an intense spring heat wave , with unusually hot temperatures from the UK and Ireland in the north, throug...

  score 0.575 | https://en.wikipedia.org/wiki/2026_European_heatwaves
  Starting in late May 202